# TTA-Torch: Dynamic Test-Time Adaptation for LLMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Griffith-7/TTA-Torch-of-Self-Tuning-Inference/blob/main/demo.ipynb)

Real-time weight updates during generation via LoRA adapters. The model adjusts its own parameters mid-generation based on its confidence level.

In [ ]:
!pip install git+https://github.com/Griffith-7/TTA-Torch-of-Self-Tuning-Inference.git

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}, Dtype: {dtype}")

## 1. Load Model and Create TTA Wrapper

In [ ]:
from tta_torch import TTAModel, load_tta_model

model, tokenizer = load_tta_model("Qwen/Qwen2.5-0.5B-Instruct", lora_rank=4)

tta = TTAModel(model, {
    "entropy_threshold": 0.4,
    "learning_rate": 1e-4,
    "inner_steps": 2,
    "max_new_tokens": 80,
    "verbose": False,
})

prompt = "What is the capital of France?"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
print(f"Prompt: {prompt}")
print(f"Input shape: {input_ids.shape}")

## 2. Compare: Baseline vs Raw TTA vs Confidence-Gated TTA

In [ ]:
prompts = [
    "What is the capital of France?",
    "What is 137 * 29?",
    "How many sides does a hexagon have?",
    "What is the chemical symbol for gold?",
    "Who wrote Romeo and Juliet?",
]

print(f"{'Method':<25} {'Prompt':<40} {'Output'}")
print("-" * 110)

for prompt in prompts:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # Baseline (frozen)
    with torch.no_grad(), model.disable_adapter():
        base_out = model.generate(input_ids, max_new_tokens=30)
    base_text = tokenizer.decode(base_out[0], skip_special_tokens=True)
    base_answer = base_text[len(prompt):].strip().split("\n")[0][:50]

    # Raw TTA
    tta.reset_weights()
    raw_out = tta.generate(input_ids, max_tokens=30)
    raw_text = tokenizer.decode(raw_out[0], skip_special_tokens=True)
    raw_answer = raw_text[len(prompt):].strip().split("\n")[0][:50]

    # Confidence-Gated TTA
    tta.reset_weights()
    cg_out, reason, entropy = tta.generate_confidence_gated(input_ids, n_passes=3, max_tokens=30)
    cg_text = tokenizer.decode(cg_out[0], skip_special_tokens=True)
    cg_answer = cg_text[len(prompt):].strip().split("\n")[0][:50]

    print(f"{'Baseline':<25} {prompt[:38]:<40} {base_answer}")
    print(f"{'Raw TTA':<25} {'':<40} {raw_answer}")
    print(f"{'Confidence-Gated (' + reason + ')':<25} {'':<40} {cg_answer}")
    print("-" * 110)

## 3. Benchmark Results

Tested on Qwen2.5-0.5B-Instruct with 40 questions (arithmetic, factual, comparison, logic) on 4GB VRAM:

| Method | Accuracy | vs Baseline |
|--------|----------|-------------|
| Baseline (greedy) | 57.5% | -- |
| TTA (raw) | 50.0% | -7.5pp |
| Self-Consistency | 62.5% | +5.0pp |
| **Confidence-Gated TTA** | **67.5%** | **+10.0pp** |

**Key finding**: Raw TTA makes the model *confident*, not *correct*. Confidence-Gated TTA fixes this by only adapting when the baseline is uncertain.